# NMRium Data Access

Retrieve the NMRium JSON payload for publicly available samples and datasets, ready for visualization in NMRium.

This notebook targets `GET /api/v1/samples/{id}/nmriumInfo` and `GET /api/v1/datasets/{id}/nmriumInfo` at `https://nmrxiv.org/api/documentation#tag/nmrium-data-access`.

In [ ]:
import os
from pprint import pprint

import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv()

BASE_URL = os.getenv("NMRXIV_BASE_URL", "https://nmrxiv.org").rstrip("/")
API_BASE = f"{BASE_URL}/api"

session = requests.Session()
session.headers.update({"Accept": "application/json", "Content-Type": "application/json"})

BASE_URL, API_BASE

In [ ]:
def api_request(method, path, **kwargs):
    url = f"{API_BASE}{path}"
    response = session.request(method, url, timeout=30, **kwargs)
    print(f"{response.request.method} {response.url} -> {response.status_code}")
    try:
        payload = response.json()
    except ValueError:
        payload = response.text
    if not response.ok:
        pprint(payload)
        response.raise_for_status()
    return payload


def get_sample_nmrium_info(sample_id):
    return api_request("GET", f"/v1/samples/{sample_id}/nmriumInfo")


def get_dataset_nmrium_info(dataset_id):
    return api_request("GET", f"/v1/datasets/{dataset_id}/nmriumInfo")


def nmrium_summary(payload):
    data = payload.get("data", {}) if isinstance(payload, dict) else {}
    spectra = data.get("spectra", []) if isinstance(data, dict) else []
    molecules = data.get("molecules", []) if isinstance(data, dict) else []
    return {
        "version": payload.get("version") if isinstance(payload, dict) else None,
        "spectra_count": len(spectra) if isinstance(spectra, list) else None,
        "molecules_count": len(molecules) if isinstance(molecules, list) else None,
    }

## Retrieve NMRium workspace data for a public sample

Accepts NMRXIV sample identifiers (`S123`), NMRXIV-prefixed identifiers (`NMRXIV:S123`), or numeric database ids. Returns `404` if the sample is not found, not public, or has no NMRium data.

In [ ]:
sample_id = "S2253"
sample_nmrium_info = get_sample_nmrium_info(sample_id)
nmrium_summary(sample_nmrium_info)

In [ ]:
pprint(sample_nmrium_info)

## Retrieve NMRium workspace data for a public dataset

Accepts NMRXIV dataset identifiers (`D123`), NMRXIV-prefixed identifiers (`NMRXIV:D123`), or numeric database ids. Returns `404` if the dataset is not found, not public, or has no NMRium data.

In [ ]:
dataset_id = "D8843"
dataset_nmrium_info = get_dataset_nmrium_info(dataset_id)
nmrium_summary(dataset_nmrium_info)

In [ ]:
pprint(dataset_nmrium_info)